# Prayer Listing PDF → JSON Converter

Converts UAE General Authority of Islamic Affairs "Prayer Listing" PDFs into JSON files.

Each PDF is expected to contain a table with these columns:

`Day | Date | Hijri Date | Fajr | Shurooq | Duhur | Asr | Maghrib | Isha`

**Output:** one JSON file per PDF, named after the month it covers (e.g. `prayers-2026-07.json`), as a list of records:

```json
{
  "date": "2026-07-01",
  "day": "Wednesday",
  "hijri": "16/1/1448",
  "Fajr": "04:03 AM",
  "Shurooq": "05:28 AM",
  "Duhur": "12:24 PM",
  "Asr": "03:42 PM",
  "Maghrib": "07:13 PM",
  "Isha": "08:38 PM"
}
```


## 1. Install & import dependencies

In [1]:
!pip install pdfplumber --break-system-packages -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import sys
import json
import re
from pathlib import Path

import pdfplumber

## 2. Configure input/output paths

Set `INPUT_PATH` to either a single PDF file or a folder containing multiple PDFs. Set `OUTPUT_DIR` to where the JSON files should be written.

In [7]:
INPUT_PATH = Path("C:\\Users\\HP\\Desktop\\Prayer")     # a single .pdf file OR a folder of .pdf files
OUTPUT_DIR = Path("C:\\Users\\HP\\Desktop\\Prayer\\data")     # where the resulting JSON files will be saved

## 3. Conversion functions

In [8]:
EXPECTED_HEADER = ["Day", "Date", "Hijri Date", "Fajr", "Shurooq", "Duhur", "Asr", "Maghrib", "Isha"]
PRAYER_COLUMNS = ["Fajr", "Shurooq", "Duhur", "Asr", "Maghrib", "Isha"]


def normalize_date(date_str: str) -> str:
    """Convert 'M/D/YYYY' -> 'YYYY-MM-DD'."""
    date_str = date_str.strip()
    month, day, year = date_str.split("/")
    return f"{int(year):04d}-{int(month):02d}-{int(day):02d}"


def extract_rows_from_pdf(pdf_path: Path):
    """Extract all table rows (across all pages) from a single PDF."""
    all_rows = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            table = page.extract_table()
            if not table:
                continue
            for row in table:
                # Skip empty rows or the header row
                if not row or row[0] is None:
                    continue
                cleaned = [c.strip() if c else "" for c in row]
                if cleaned[:3] == EXPECTED_HEADER[:3] or cleaned[0] == "Day":
                    continue
                # A valid data row should have a date-like value in column 1
                if re.match(r"^\d{1,2}/\d{1,2}/\d{4}$", cleaned[1] or ""):
                    all_rows.append(cleaned)
    return all_rows


def rows_to_records(rows):
    """Convert raw table rows into structured dicts."""
    records = []
    for row in rows:
        day, date_str, hijri, *times = row
        if len(times) < len(PRAYER_COLUMNS):
            # Skip malformed rows (e.g. extraction glitches)
            continue
        record = {
            "date": normalize_date(date_str),
            "day": day,
            "hijri": hijri,
        }
        for name, value in zip(PRAYER_COLUMNS, times):
            record[name] = value
        records.append(record)
    return records


def convert_pdf(pdf_path: Path, output_dir: Path) -> Path:
    rows = extract_rows_from_pdf(pdf_path)
    if not rows:
        raise ValueError(f"No prayer table rows found in {pdf_path.name}")

    records = rows_to_records(rows)
    records.sort(key=lambda r: r["date"])

    # Derive output filename from the first record's year-month
    first_date = records[0]["date"]  # YYYY-MM-DD
    year_month = first_date[:7]      # YYYY-MM
    output_path = output_dir / f"prayers-{year_month}.json"

    output_dir.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2, ensure_ascii=False)

    return output_path


def find_pdfs(input_path: Path):
    if input_path.is_file():
        if input_path.suffix.lower() != ".pdf":
            raise ValueError(f"{input_path} is not a PDF file")
        return [input_path]
    if input_path.is_dir():
        pdfs = sorted(input_path.glob("*.pdf"))
        if not pdfs:
            raise ValueError(f"No PDF files found in {input_path}")
        return pdfs
    raise FileNotFoundError(f"{input_path} does not exist")

## 4. Run the conversion

In [9]:
pdfs = find_pdfs(INPUT_PATH)

results = []
for pdf_path in pdfs:
    try:
        output_path = convert_pdf(pdf_path, OUTPUT_DIR)
        print(f"✓ {pdf_path.name} -> {output_path}")
        results.append((pdf_path, output_path))
    except Exception as e:
        print(f"✗ {pdf_path.name}: {e}")

✓ 20260705_111519.pdf -> C:\Users\HP\Desktop\Prayer\data\prayers-2026-07.json
✓ 20260705_113351.pdf -> C:\Users\HP\Desktop\Prayer\data\prayers-2026-08.json
✓ 20260705_113357.pdf -> C:\Users\HP\Desktop\Prayer\data\prayers-2026-09.json
✓ 20260705_113403.pdf -> C:\Users\HP\Desktop\Prayer\data\prayers-2026-10.json
✓ 20260705_113408.pdf -> C:\Users\HP\Desktop\Prayer\data\prayers-2026-11.json
✓ 20260705_113413.pdf -> C:\Users\HP\Desktop\Prayer\data\prayers-2026-12.json
✓ 20260705_113446.pdf -> C:\Users\HP\Desktop\Prayer\data\prayers-2027-01.json
✓ 20260705_113450.pdf -> C:\Users\HP\Desktop\Prayer\data\prayers-2027-02.json
✓ 20260705_113453.pdf -> C:\Users\HP\Desktop\Prayer\data\prayers-2027-03.json
✓ 20260705_113457.pdf -> C:\Users\HP\Desktop\Prayer\data\prayers-2027-04.json
✓ 20260705_113501.pdf -> C:\Users\HP\Desktop\Prayer\data\prayers-2027-05.json
✓ 20260705_113508.pdf -> C:\Users\HP\Desktop\Prayer\data\prayers-2027-06.json


## 5. Preview a result

Quickly sanity-check the first and last rows of the most recently generated JSON file.

In [10]:
if results:
    _, last_output = results[-1]
    with open(last_output) as f:
        data = json.load(f)
    print(f"{len(data)} records in {last_output.name}")
    print(json.dumps(data[0], indent=2))
    print(json.dumps(data[-1], indent=2))
else:
    print("No files were converted.")

5 records in prayers-2027-06.json
{
  "date": "2027-06-01",
  "day": "Tuesday",
  "hijri": "26/12/1448",
  "Fajr": "04:01 AM",
  "Shurooq": "05:24 AM",
  "Duhur": "12:18 PM",
  "Asr": "03:37 PM",
  "Maghrib": "07:05 PM",
  "Isha": "08:29 PM"
}
{
  "date": "2027-06-05",
  "day": "Saturday",
  "hijri": "30/12/1448",
  "Fajr": "04:00 AM",
  "Shurooq": "05:23 AM",
  "Duhur": "12:18 PM",
  "Asr": "03:37 PM",
  "Maghrib": "07:06 PM",
  "Isha": "08:31 PM"
}
